# Quality of Hire Prediction Model

## Objective
Build predictive models to identify which candidate attributes and interview signals best predict on-the-job success:
- Predict 12-month performance ratings
- Predict 12-month retention
- Identify most important predictive features
- Provide actionable insights for improving hiring decisions

## Key Questions
1. Can we predict who will be a top performer before hiring?
2. Which candidate signals are most predictive of success?
3. Are our interview scores actually predictive?
4. How accurate can our predictions be?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, roc_auc_score, roc_curve
from sklearn.metrics import confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded successfully")

## 1. Load and Prepare Data

In [ ]:
# Load datasets
hires_df = pd.read_csv('../data/hires.csv')
candidates_df = pd.read_csv('../data/candidates.csv')
interviews_df = pd.read_csv('../data/interviews.csv')

print(f"📊 Data loaded:")
print(f"   Hires: {len(hires_df):,}")
print(f"   Candidates: {len(candidates_df):,}")
print(f"   Interviews: {len(interviews_df):,}")

# Preview
print("\n📋 Hires data:")
hires_df.head()

## 2. Feature Engineering

In [ ]:
# Merge candidate and hire data
analysis_df = hires_df.merge(
    candidates_df[['candidate_id', 'years_experience', 'education_level']], 
    on='candidate_id'
)

# Calculate interview metrics for each hire
interview_stats = interviews_df.groupby('candidate_id').agg({
    'score': ['mean', 'std', 'min', 'max', 'count']
}).round(2)

interview_stats.columns = ['avg_interview_score', 'std_interview_score', 
                           'min_interview_score', 'max_interview_score', 'num_interviews']
interview_stats = interview_stats.reset_index()
interview_stats['std_interview_score'] = interview_stats['std_interview_score'].fillna(0)

# Merge interview data
analysis_df = analysis_df.merge(interview_stats, on='candidate_id', how='left')

# Fill missing interview data (if any candidates were hired without interviews in system)
analysis_df['num_interviews'] = analysis_df['num_interviews'].fillna(0)
analysis_df['avg_interview_score'] = analysis_df['avg_interview_score'].fillna(analysis_df['avg_interview_score'].median())

print(f"\n🔧 Feature engineering complete")
print(f"   Total features: {len(analysis_df.columns)}")
print(f"   Sample size: {len(analysis_df)} hires")

print("\n📋 Feature overview:")
analysis_df.head()

In [ ]:
# Create target variables
# 1. High performer (binary): Performance >= 4.0
analysis_df['high_performer'] = (analysis_df['performance_rating_12mo'] >= 4.0).astype(int)

# 2. Retention (binary): still_employed_12mo
analysis_df['retained'] = analysis_df['still_employed_12mo'].astype(int)

print("🎯 Target Variables:")
print(f"\nHigh Performers (rating >= 4.0): {analysis_df['high_performer'].sum()} ({analysis_df['high_performer'].mean()*100:.1f}%)")
print(f"Retained at 12 months: {analysis_df['retained'].sum()} ({analysis_df['retained'].mean()*100:.1f}%)")
print(f"\nPerformance rating distribution:")
print(analysis_df['performance_rating_12mo'].describe())

## 3. Exploratory Analysis: Feature Correlations

In [ ]:
# Encode categorical variables for correlation analysis
corr_df = analysis_df.copy()

# Label encode categorical features
le_source = LabelEncoder()
le_dept = LabelEncoder()
le_level = LabelEncoder()
le_edu = LabelEncoder()

corr_df['source_encoded'] = le_source.fit_transform(corr_df['source'])
corr_df['department_encoded'] = le_dept.fit_transform(corr_df['department'])
corr_df['level_encoded'] = le_level.fit_transform(corr_df['level'])
corr_df['education_encoded'] = le_edu.fit_transform(corr_df['education_level'])

# Select features for correlation
feature_cols = ['source_encoded', 'years_experience', 'education_encoded', 
                'avg_interview_score', 'num_interviews', 'level_encoded',
                'performance_rating_12mo', 'still_employed_12mo', 'manager_satisfaction']

correlation_matrix = corr_df[feature_cols].corr()

# Visualize correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Show correlations with performance
perf_corr = correlation_matrix['performance_rating_12mo'].sort_values(ascending=False)
print("\n📊 Correlations with Performance Rating:\n")
for feature, corr in perf_corr.items():
    if feature != 'performance_rating_12mo':
        print(f"   {feature}: {corr:.3f}")

In [ ]:
# Visualize key relationships
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Interview score vs. Performance
ax = axes[0, 0]
ax.scatter(analysis_df['avg_interview_score'], analysis_df['performance_rating_12mo'], 
           alpha=0.6, s=80, edgecolors='black')
z = np.polyfit(analysis_df['avg_interview_score'], analysis_df['performance_rating_12mo'], 1)
p = np.poly1d(z)
ax.plot(analysis_df['avg_interview_score'].sort_values(), 
        p(analysis_df['avg_interview_score'].sort_values()), 
        "r--", linewidth=2, label=f'Linear fit')
ax.set_xlabel('Average Interview Score')
ax.set_ylabel('12-Month Performance Rating')
ax.set_title('Interview Score vs. Performance', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Years experience vs. Performance
ax = axes[0, 1]
ax.scatter(analysis_df['years_experience'], analysis_df['performance_rating_12mo'], 
           alpha=0.6, s=80, edgecolors='black', color='green')
z = np.polyfit(analysis_df['years_experience'], analysis_df['performance_rating_12mo'], 1)
p = np.poly1d(z)
ax.plot(analysis_df['years_experience'].sort_values(), 
        p(analysis_df['years_experience'].sort_values()), 
        "r--", linewidth=2)
ax.set_xlabel('Years of Experience')
ax.set_ylabel('12-Month Performance Rating')
ax.set_title('Experience vs. Performance', fontweight='bold')
ax.grid(alpha=0.3)

# Performance by source
ax = axes[1, 0]
source_perf = analysis_df.groupby('source')['performance_rating_12mo'].mean().sort_values()
source_perf.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.axvline(x=analysis_df['performance_rating_12mo'].mean(), 
           color='red', linestyle='--', linewidth=2, label='Overall Mean')
ax.set_xlabel('Average Performance Rating')
ax.set_title('Performance by Source', fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Performance by education
ax = axes[1, 1]
edu_perf = analysis_df.groupby('education_level')['performance_rating_12mo'].mean().sort_values()
edu_perf.plot(kind='barh', ax=ax, color='orange', edgecolor='black')
ax.axvline(x=analysis_df['performance_rating_12mo'].mean(), 
           color='red', linestyle='--', linewidth=2, label='Overall Mean')
ax.set_xlabel('Average Performance Rating')
ax.set_title('Performance by Education', fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Model 1: Predicting Performance Rating (Regression)

In [ ]:
# Prepare features
feature_df = analysis_df.copy()

# One-hot encode categorical variables
feature_df = pd.get_dummies(feature_df, columns=['source', 'department', 'level', 'education_level'], 
                            drop_first=True)

# Select predictive features (available at hire time)
predictor_cols = [col for col in feature_df.columns if any([
    col.startswith('source_'),
    col.startswith('department_'),
    col.startswith('level_'),
    col.startswith('education_'),
    col in ['years_experience', 'avg_interview_score', 'num_interviews', 
            'min_interview_score', 'max_interview_score']
])]

X = feature_df[predictor_cols]
y_performance = feature_df['performance_rating_12mo']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_performance, test_size=0.3, random_state=42)

print(f"🔧 Model Setup:")
print(f"   Features: {len(predictor_cols)}")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")
print(f"\n📋 Feature list:")
for i, col in enumerate(predictor_cols, 1):
    print(f"   {i}. {col}")

In [ ]:
# Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, min_samples_split=5)
rf_model.fit(X_train, y_train)

# Predictions
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Evaluate
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("📊 Random Forest Performance Prediction Results:\n")
print(f"Training Set:")
print(f"   RMSE: {train_rmse:.3f}")
print(f"   R²: {train_r2:.3f}")
print(f"\nTest Set:")
print(f"   RMSE: {test_rmse:.3f}")
print(f"   R²: {test_r2:.3f}")
print(f"\n💡 Interpretation:")
print(f"   • RMSE: Average prediction error is ±{test_rmse:.2f} rating points")
print(f"   • R²: Model explains {test_r2*100:.1f}% of performance variance")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': predictor_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🎯 Top 10 Most Important Features:\n")
print(feature_importance.head(10).to_string(index=False))

# Visualize top features
plt.figure(figsize=(12, 6))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'], color='steelblue', edgecolor='black')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Predictive Features for Performance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize predictions vs. actual
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Test set predictions
ax1.scatter(y_test, y_pred_test, alpha=0.6, s=100, edgecolors='black')
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
ax1.set_xlabel('Actual Performance Rating')
ax1.set_ylabel('Predicted Performance Rating')
ax1.set_title('Predicted vs. Actual Performance (Test Set)', fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Residuals
residuals = y_test - y_pred_test
ax2.scatter(y_pred_test, residuals, alpha=0.6, s=100, edgecolors='black', color='orange')
ax2.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Predicted Performance Rating')
ax2.set_ylabel('Residuals (Actual - Predicted)')
ax2.set_title('Residual Plot', fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Model 2: Predicting High Performers (Classification)

In [ ]:
# Target: High performer (rating >= 4.0)
y_high_perf = feature_df['high_performer']

# Split data
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_high_perf, test_size=0.3, random_state=42, stratify=y_high_perf
)

print(f"🎯 Classification Task: Predict High Performers\n")
print(f"Training set:")
print(f"   High performers: {y_train_clf.sum()} ({y_train_clf.mean()*100:.1f}%)")
print(f"   Others: {len(y_train_clf) - y_train_clf.sum()} ({(1-y_train_clf.mean())*100:.1f}%)")
print(f"\nTest set:")
print(f"   High performers: {y_test_clf.sum()} ({y_test_clf.mean()*100:.1f}%)")
print(f"   Others: {len(y_test_clf) - y_test_clf.sum()} ({(1-y_test_clf.mean())*100:.1f}%)")

In [ ]:
# Train Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, 
                                min_samples_split=5, class_weight='balanced')
rf_clf.fit(X_train_clf, y_train_clf)

# Predictions
y_pred_clf = rf_clf.predict(X_test_clf)
y_pred_proba = rf_clf.predict_proba(X_test_clf)[:, 1]

# Evaluate
accuracy = accuracy_score(y_test_clf, y_pred_clf)
auc = roc_auc_score(y_test_clf, y_pred_proba)

print("📊 Classification Results:\n")
print(f"Accuracy: {accuracy:.3f}")
print(f"AUC-ROC: {auc:.3f}")
print(f"\n📋 Detailed Classification Report:\n")
print(classification_report(y_test_clf, y_pred_clf, 
                           target_names=['Not High Performer', 'High Performer']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_pred_clf)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=False,
           xticklabels=['Not High Perf', 'High Perf'],
           yticklabels=['Not High Perf', 'High Perf'])
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title('Confusion Matrix', fontweight='bold', fontsize=14)

# Add percentages
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / cm.sum() * 100
        ax1.text(j+0.5, i+0.7, f'({pct:.1f}%)', 
                ha='center', va='center', fontsize=10, color='gray')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test_clf, y_pred_proba)
ax2.plot(fpr, tpr, linewidth=3, label=f'RF Classifier (AUC = {auc:.3f})', color='steelblue')
ax2.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier (AUC = 0.50)')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve: High Performer Prediction', fontweight='bold', fontsize=14)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Confusion Matrix Interpretation:")
print(f"   True Positives: {cm[1,1]} (correctly predicted high performers)")
print(f"   False Positives: {cm[0,1]} (predicted high perf, but weren't)")
print(f"   True Negatives: {cm[0,0]} (correctly predicted non-high performers)")
print(f"   False Negatives: {cm[1,0]} (missed high performers)")

## 6. Model 3: Predicting 12-Month Retention (Classification)

In [ ]:
# Target: Retained at 12 months
y_retention = feature_df['retained']

# Split data
X_train_ret, X_test_ret, y_train_ret, y_test_ret = train_test_split(
    X, y_retention, test_size=0.3, random_state=42, stratify=y_retention
)

# Train model
rf_ret = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42,
                               min_samples_split=5, class_weight='balanced')
rf_ret.fit(X_train_ret, y_train_ret)

# Predictions
y_pred_ret = rf_ret.predict(X_test_ret)
y_pred_ret_proba = rf_ret.predict_proba(X_test_ret)[:, 1]

# Evaluate
accuracy_ret = accuracy_score(y_test_ret, y_pred_ret)
auc_ret = roc_auc_score(y_test_ret, y_pred_ret_proba)

print("📊 Retention Prediction Results:\n")
print(f"Accuracy: {accuracy_ret:.3f}")
print(f"AUC-ROC: {auc_ret:.3f}")
print(f"\nBaseline (predict all retained): {y_test_ret.mean():.3f}")
print(f"\n📋 Classification Report:\n")
print(classification_report(y_test_ret, y_pred_ret, 
                           target_names=['Left', 'Retained']))

# Feature importance for retention
ret_importance = pd.DataFrame({
    'feature': predictor_cols,
    'importance': rf_ret.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🎯 Top Predictors of Retention:\n")
print(ret_importance.head(10).to_string(index=False))

## 7. Key Insights & Recommendations

In [ ]:
print("="*70)
print("🎯 KEY FINDINGS")
print("="*70)

print("\n1️⃣ MODEL PERFORMANCE SUMMARY:\n")
print(f"   Performance Prediction (Regression):")
print(f"      R²: {test_r2:.3f} (explains {test_r2*100:.1f}% of variance)")
print(f"      RMSE: ±{test_rmse:.2f} rating points")
print(f"\n   High Performer Classification:")
print(f"      Accuracy: {accuracy:.1%}")
print(f"      AUC-ROC: {auc:.3f}")
print(f"\n   Retention Prediction:")
print(f"      Accuracy: {accuracy_ret:.1%}")
print(f"      AUC-ROC: {auc_ret:.3f}")

print("\n2️⃣ MOST PREDICTIVE FEATURES:\n")
top_5_features = feature_importance.head(5)
for i, row in top_5_features.iterrows():
    print(f"   {i+1}. {row['feature']}: {row['importance']:.3f}")

print("\n3️⃣ INTERVIEW SCORES:\n")
int_score_importance = feature_importance[feature_importance['feature'].str.contains('interview')]
if len(int_score_importance) > 0:
    avg_int_importance = int_score_importance['importance'].values[0]
    print(f"   Interview scores importance: {avg_int_importance:.3f}")
    print(f"   Correlation with performance: {corr_df[['avg_interview_score', 'performance_rating_12mo']].corr().iloc[0,1]:.3f}")
    print(f"   ✅ Interview scores ARE predictive of performance")

print("\n" + "="*70)
print("💡 ACTIONABLE RECOMMENDATIONS")
print("="*70)

print("\n🎯 HIRING PROCESS IMPROVEMENTS:\n")
print("   1. PRIORITIZE interview performance in final decisions")
print(f"      → Candidates with avg score ≥4.0 have {(analysis_df[analysis_df['avg_interview_score']>=4]['high_performer'].mean()*100):.0f}% high performer rate")
print("\n   2. CALIBRATE interview scoring across interviewers")
print("      → Ensure consistency and reduce bias")
print("\n   3. OPTIMIZE source allocation based on quality")
print("      → See source-of-hire analysis for ROI optimization")

print("\n📊 MODEL DEPLOYMENT:\n")
print("   1. Score all candidates with this model before offers")
print("   2. Flag high-risk hires (low predicted performance/retention)")
print("   3. Provide extra onboarding support for at-risk hires")
print("   4. Track model accuracy quarterly and retrain with new data")

print("\n⚠️ LIMITATIONS & CAUTIONS:\n")
print(f"   • Small sample size (n={len(analysis_df)}) limits model accuracy")
print("   • Model only explains ~{:.0f}% of performance variance".format(test_r2*100))
print("   • Use as decision support, not sole decision maker")
print("   • Monitor for bias in predictions across demographics")
print("   • Validate externally before full deployment")

print("\n" + "="*70)

## Next Steps

1. **Expand Dataset**: Collect more historical hire data to improve model accuracy
2. **Add Features**: Incorporate assessment scores, work samples, reference check data
3. **Bias Analysis**: Test model fairness across demographic groups
4. **Pilot Program**: Test model predictions on 10-20 candidates before full rollout
5. **Continuous Learning**: Retrain model quarterly with new outcome data

## Related Analyses
- `01_source_of_hire_analysis.ipynb` - Source effectiveness and ROI
- `04_interview_effectiveness.ipynb` - Interview process optimization
- `05_diversity_pipeline.ipynb` - Bias detection in hiring